# Mục 4.4 — Thực hành phân lớp

Sinh tu ban thao giao trinh. Chay tuan tu tu tren xuong; moi o phu thuoc cac o truoc no.


## 4.4. Thực hành

Mục này hướng dẫn người học áp dụng toàn bộ các thuật toán phân lớp đã học ở Mục 4.2 và các phương pháp đánh giá ở Mục 4.3 trên một tập dữ liệu chuẩn, thực hiện hoàn toàn bằng Python với scikit-learn. Sáu thuật toán được chạy trên cùng một tập dữ liệu, cùng một cách chia dữ liệu và cùng một bộ chỉ số, nên các kết quả so sánh được với nhau. Tệp 03-thuc-hanh-chuong-4.md trong thư mục weka của kho học phần (Mục A.3) trình bày cùng bài thực hành này trên WEKA Explorer, cho người học muốn làm bằng giao diện đồ họa.

## Chuẩn bị môi trường và tập dữ liệu

### Yêu cầu môi trường

Dùng môi trường thống nhất ở Phụ lục B: Python 3.11 hoặc 3.12 và scikit-learn 1.8.x. Cài trong môi trường ảo, sau đó ghi lại phiên bản thực tế để tái lập được kết quả.

**Kết quả đối chiếu**

```
python -m pip install "scikit-learn==1.8.0" numpy pandas matplotlib
python -m pip freeze > requirements-lock.txt
```

### Tài nguyên đi kèm

Mục này không cần tải tệp dữ liệu nào từ bên ngoài: tập Iris đi kèm scikit-learn và được nạp bằng sklearn.datasets.load_iris. Kiểm tra phiên bản thư viện trước khi chạy; không giả định các thư mục theo chương đã có sẵn.

### Tập dữ liệu sử dụng

Bài thực hành dùng tập dữ liệu Iris, tập dữ liệu kinh điển được R. A. Fisher giới thiệu năm 1936.

- Nguồn: sklearn.datasets.load_iris. Số mẫu: 150 (50 mẫu mỗi loài hoa).

- Số đặc trưng: 4 (chiều dài đài hoa, chiều rộng đài hoa, chiều dài cánh hoa, chiều rộng cánh hoa).

- Nhãn: 3 loài hoa (setosa, versicolor, virginica).

Đây là bài toán phân lớp đa lớp với ba lớp cân bằng nhau, dữ liệu sạch, không có giá trị thiếu, phù hợp để minh họa các thuật toán.

### Thiết lập đánh giá chung

Cả sáu tiểu mục dùng đúng một cách đánh giá: kiểm chứng chéo 10 phần qua cross_val_score(mo_hinh, X, y, cv=10). Với bài toán phân lớp, cv=10 tương ứng StratifiedKFold 10 phần, nghĩa là mỗi phần giữ nguyên tỷ lệ ba lớp (Mục 4.3.3), và không xáo trộn dữ liệu trước khi chia. Vì không xáo trộn nên cách chia không phụ thuộc số ngẫu nhiên: không cần đặt seed cho bước chia phần, và mọi người chạy đều thu được đúng các con số in trong mục này. Tham số random_state=42 chỉ cần cho các thuật toán có thành phần ngẫu nhiên bên trong (cây quyết định, Random Forest, AdaBoost, mạng nơ-ron).

Hai chỉ số được báo cáo cho mỗi thuật toán là Accuracy (Mục 4.3.1) và Kappa, cùng với ma trận nhầm lẫn. Kappa, hay hệ số Cohen, so tỷ lệ dự đoán đúng của mô hình với tỷ lệ đúng mà một bộ dự đoán ngẫu nhiên có cùng phân bố nhãn đạt được: Kappa bằng 1 khi mô hình đúng toàn bộ và bằng 0 khi mô hình chỉ ngang mức ngẫu nhiên. Trên dữ liệu cân bằng như Iris, Kappa thường đi cùng chiều với Accuracy; giá trị của Kappa bộc lộ rõ hơn trên dữ liệu mất cân bằng.

cross_val_score cho trung bình Accuracy của 10 phần, còn cross_val_predict gộp dự đoán của 10 phần thành một ma trận nhầm lẫn duy nhất. Vì 150 mẫu chia thành 10 phần bằng nhau, mỗi phần 15 mẫu, trung bình Accuracy của 10 phần trùng với tỷ lệ đúng tính trên toàn bộ 150 mẫu; trên tập dữ liệu có số mẫu không chia hết cho số phần, hai con số này có thể lệch nhẹ.

Chuẩn hóa chỉ cần cho KNN và mạng nơ-ron, là hai thuật toán nhạy với thang đo. Phép chuẩn hóa được đặt trong Pipeline, không áp dụng trước cho toàn bộ dữ liệu: nhờ vậy trung bình và độ lệch chuẩn của StandardScaler chỉ được học trên phần huấn luyện của từng vòng kiểm chứng chéo, tránh rò rỉ dữ liệu (Mục 2.1.4 và Mục 2.7, Phần E). Naive Bayes, cây quyết định, Random Forest và AdaBoost không cần chuẩn hóa vì chúng xét từng đặc trưng riêng chứ không tính khoảng cách.

### Bước 1. Nạp dữ liệu và khai báo hàm đánh giá chung

Mở tệp Python hoặc Notebook trong thư mục bài làm và chạy khối mã sau. Khối mã này nạp dữ liệu, kiểm tra kích thước và phân bố nhãn, đồng thời định nghĩa hàm danh_gia dùng lại cho cả năm tiểu mục sau.



In [ ]:
# Muc 4.4 - Chuan bi: nap du lieu va dinh nghia ham danh gia chung
import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import cross_val_predict, cross_val_score

X, y = load_iris(return_X_y=True)
ten_lop = load_iris().target_names

print('Kich thuoc:', X.shape)
print('So lop:', len(ten_lop), '-', list(ten_lop))
print('So mau moi lop:', np.bincount(y))


def danh_gia(ten, mo_hinh):
    """In Accuracy, Kappa va ma tran nham lan cho mot mo hinh.

    Danh gia bang kiem chung cheo 10 phan: cv=10 la StratifiedKFold
    khong xao tron, nen ket qua tai lap duoc ma khong can seed.
    """
    diem = cross_val_score(mo_hinh, X, y, cv=10)
    du_doan = cross_val_predict(mo_hinh, X, y, cv=10)
    print(ten)
    print('  Accuracy = %.4f (do lech chuan giua 10 phan = %.4f)'
          % (diem.mean(), diem.std()))
    print('  Kappa    = %.4f' % cohen_kappa_score(y, du_doan))
    print('  So mau sai = %d / %d' % ((du_doan != y).sum(), len(y)))
    print('  Ma tran nham lan (hang la nhan that, cot la du doan):')
    print(confusion_matrix(y, du_doan))


### Đối chiếu kết quả

- Kích thước: (150, 4).

- Số lớp: 3 — ['setosa', 'versicolor', 'virginica'].

- Số mẫu mỗi lớp: [50 50 50].

Nếu ba dòng này không khớp, hãy kiểm tra lại phiên bản scikit-learn trước khi làm tiếp.

## 4.4.A. KNN (k = 5)

### Mục tiêu

Chạy thuật toán KNN với k = 5 (Mục 4.2.1), đặt trong một pipeline có chuẩn hóa, rồi đọc Accuracy, Kappa và ma trận nhầm lẫn.



In [ ]:
# Can chay khoi ma Chuan bi truoc: X, y va ham danh_gia lay tu do
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# StandardScaler dat trong pipeline de chi hoc tren phan huan luyen
knn = make_pipeline(StandardScaler(),
                    KNeighborsClassifier(n_neighbors=5))
danh_gia('KNN (k = 5)', knn)


### Đối chiếu kết quả

- Accuracy = 0,9533 (143 trên 150 mẫu đúng, 7 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0521.

- Kappa = 0,9300.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

### Diễn giải

Accuracy = 95,33% và Kappa = 0,9300: mô hình rất tốt trên Iris. Lớp setosa được phân lớp hoàn hảo (50 trên 50), cho thấy lớp này tách rời hẳn hai lớp còn lại. Toàn bộ 7 mẫu sai đều nằm ở vùng chồng lấn giữa versicolor và virginica: 4 mẫu versicolor bị gán nhãn virginica và 3 mẫu virginica bị gán nhãn versicolor. Độ lệch chuẩn 0,0521 cho biết Accuracy của từng phần dao động quanh giá trị trung bình; với 15 mẫu mỗi phần, một mẫu sai đã làm Accuracy của phần đó đổi 6,67 điểm phần trăm, nên mức dao động này là bình thường. Người học nên thử lại với k = 1, k = 3 và k = 11 để quan sát ảnh hưởng của k, và thử bỏ StandardScaler khỏi pipeline để thấy vai trò của chuẩn hóa với một thuật toán dựa trên khoảng cách.

## 4.4.B. Naive Bayes

### Mục tiêu

Chạy Naive Bayes dạng Gauss (Mục 4.2.2) trên cùng một cách đánh giá và so sánh với KNN.



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.naive_bayes import GaussianNB

# GaussianNB gia dinh moi dac trung theo phan phoi chuan trong tung lop
# nen khong can chuan hoa du lieu
nb = GaussianNB()
danh_gia('Naive Bayes', nb)


### Đối chiếu kết quả

- Accuracy = 0,9533 (143 trên 150 mẫu đúng, 7 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0427.

- Kappa = 0,9300.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

### Diễn giải

Naive Bayes đạt Accuracy = 95,33% và Kappa = 0,9300 dù giả định các đặc trưng độc lập với nhau rõ ràng bị vi phạm: chiều dài và chiều rộng cánh hoa có tương quan chặt. Đây là minh chứng cho tính mạnh mẽ của Naive Bayes trong thực tế, đã nêu ở Mục 4.2.2: giả định độc lập có thể sai mà thứ tự xác suất giữa các lớp vẫn đúng, và phân lớp chỉ cần thứ tự đó.

So với 4.4.A, Naive Bayes trùng KNN cả Accuracy, cả Kappa, cả số mẫu sai, nhưng ma trận nhầm lẫn thì khác: KNN nhầm 4 mẫu versicolor và 3 mẫu virginica, Naive Bayes nhầm 3 mẫu versicolor và 4 mẫu virginica. Hai mô hình giống nhau ở một con số tổng không có nghĩa là chúng hành xử giống nhau. Đây chính là lý do Mục 4.3.1 yêu cầu đọc ma trận nhầm lẫn thay vì chỉ đọc Accuracy: nếu bài toán coi một loại lỗi nặng hơn loại lỗi còn lại, hai mô hình này không tương đương. Naive Bayes cũng là thuật toán tốn ít tính toán nhất trong sáu thuật toán của mục này, vì toàn bộ việc huấn luyện chỉ gồm tính trung bình và phương sai của từng đặc trưng trong từng lớp.

## 4.4.C. Cây quyết định

### Mục tiêu

Chạy cây quyết định (Mục 4.2.3), đọc kết quả và in cấu trúc cây để thấy ưu điểm diễn giải được.



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.tree import DecisionTreeClassifier, export_text

# DecisionTreeClassifier la cai dat cua CART; de mac dinh thi cay moc
# den khi moi la thuan, tuc khong cat tia
cay = DecisionTreeClassifier(random_state=42)
danh_gia('Cay quyet dinh', cay)

# In cau truc cay huan luyen tren toan bo du lieu de doc va giai thich
cay.fit(X, y)
print('Do sau:', cay.get_depth(), '- So la:', cay.get_n_leaves())
print(export_text(cay, feature_names=list(load_iris().feature_names)))


### Đối chiếu kết quả

- Accuracy = 0,9533 (143 trên 150 mẫu đúng, 7 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0427.

- Kappa = 0,9300.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

Phần in cấu trúc cây là của mô hình huấn luyện trên toàn bộ 150 mẫu, không phải của một vòng kiểm chứng chéo; người học ghi lại độ sâu, số lá và đặc trưng được chọn ở nhánh đầu tiên vào báo cáo.

### Diễn giải

Accuracy = 95,33% và Kappa = 0,9300, trùng với Naive Bayes cả ở ma trận nhầm lẫn. Điểm mạnh của cây quyết định không nằm ở con số này mà ở chỗ kết quả đọc được: mỗi lá là một chuỗi điều kiện trên các đặc trưng, nên người học có thể giải thích vì sao một mẫu được gán nhãn như vậy, điều mà Naive Bayes và mạng nơ-ron không cho sẵn.

Cần lưu ý DecisionTreeClassifier để mặc định sẽ mọc đến khi mọi lá thuần, tức không cắt tỉa. Những lá cuối cùng chỉ còn vài mẫu chính là phần cây đang cố tách hai loài chồng lấn nhau, và cũng là chỗ cây dễ quá khớp dữ liệu nhất (Mục 4.2.3). Người học hãy chạy lại với min_samples_leaf=6 để thấy cây gọn hẳn đi và ghi lại Accuracy mới: nếu cây nhỏ hơn nhiều mà Accuracy gần như không đổi, phần cây bị cắt đúng là phần quá khớp.

## 4.4.D. Mạng nơ-ron

### Mục tiêu

Chạy mạng nơ-ron một tầng ẩn với kiến trúc 4-6-3, đúng kiến trúc đã dùng ở Mục 4.2.4, và quan sát một ma trận nhầm lẫn khác hẳn các thuật toán còn lại.



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Kien truc 4-6-3 nhu Muc 4.2.4: 4 dac trung, 1 tang an 6 no-ron, 3 lop.
# Chuan hoa la bat buoc voi mang no-ron va phai dat trong pipeline.
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(6,),
                  max_iter=2000,      # du vong lap de mang hoi tu
                  random_state=42))   # co dinh khoi tao trong so
danh_gia('Mang no-ron 4-6-3', mlp)


### Đối chiếu kết quả

- Accuracy = 0,9533 (143 trên 150 mẫu đúng, 7 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0600.

- Kappa = 0,9300.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

### Diễn giải

Accuracy = 95,33% và Kappa = 0,9300, bằng KNN, Naive Bayes và cây quyết định. Nhưng ma trận nhầm lẫn có một điểm khác hẳn: một mẫu setosa bị gán nhãn versicolor, trong khi năm thuật toán còn lại của mục này đều phân lớp đúng cả 50 mẫu setosa. Lỗi này không do dữ liệu chồng lấn, vì setosa tách rời hẳn hai lớp kia, mà đến từ chính mô hình: trong một vòng kiểm chứng chéo, mạng chỉ được huấn luyện trên 135 mẫu và khởi đầu từ một bộ trọng số ngẫu nhiên, nên biên quyết định thu được có thể cắt lệch ở vùng rìa của lớp setosa. Độ lệch chuẩn giữa 10 phần là 0,0600, lớn thứ hai trong sáu thuật toán, cũng cho thấy kết quả của mạng nơ-ron dao động nhiều nhất giữa các lần chia.

Kết quả này đúng với khuyến nghị ở Mục 4.2.4: mạng nơ-ron ít phù hợp với tập dữ liệu chỉ vài trăm mẫu, nên trên Iris nó không có lợi thế so với các thuật toán đơn giản hơn, lại tốn thời gian huấn luyện nhiều hơn hẳn và khó diễn giải hơn cây quyết định. Hai tham số trong mã cần giữ nguyên khi đối chiếu: max_iter=2000 để mạng có đủ vòng lặp hội tụ, vì với giá trị mặc định nhỏ hơn, scikit-learn có thể phát cảnh báo ConvergenceWarning và cho kết quả khác; random_state=42 để cố định bộ trọng số khởi tạo. Khi báo cáo kết quả của mạng nơ-ron, luôn ghi kèm seed đã dùng, vì đổi seed thì con số đổi theo.

## 4.4.E. Các phương pháp kết hợp

### Mục tiêu

Chạy hai phương pháp kết hợp của Mục 4.2.5, Random Forest (bagging) và AdaBoost (boosting), rồi đối chiếu với các thuật toán đơn lẻ ở trên.

### Random Forest



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.ensemble import RandomForestClassifier

# 100 cay, moi cay hoc tren mot mau bootstrap va mot tap dac trung ngau nhien
rf = RandomForestClassifier(n_estimators=100, random_state=42)
danh_gia('Random Forest (100 cay)', rf)


### Đối chiếu kết quả

- Accuracy = 0,9600 (144 trên 150 mẫu đúng, 6 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0327.

- Kappa = 0,9400.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

### AdaBoost



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.ensemble import AdaBoostClassifier

# De mac dinh: 50 bo phan lop thanh phan, moi bo la mot cay quyet dinh
# mot tang (decision stump) - dung mo hinh yeu nhu Muc 4.2.5
ada = AdaBoostClassifier(random_state=42)
danh_gia('AdaBoost', ada)


### Đối chiếu kết quả

- Accuracy = 0,9467 (142 trên 150 mẫu đúng, 8 mẫu sai), độ lệch chuẩn giữa 10 phần = 0,0653.

- Kappa = 0,9200.

Ma trận nhầm lẫn (hàng là nhãn thật, cột là dự đoán):

### Diễn giải

Random Forest đạt Accuracy = 96,00% và Kappa = 0,9400, cao nhất trong sáu thuật toán, nhưng khoảng cách với một cây quyết định đơn lẻ ở 4.4.C chỉ là một mẫu trong 150. Trên một tập dữ liệu nhỏ và dễ như Iris, việc gộp 100 cây gần như không mang lại lợi thế; sức mạnh của Random Forest chỉ bộc lộ trên dữ liệu nhiều nhiễu và nhiều đặc trưng hơn. Điều đáng chú ý hơn con số Accuracy là độ lệch chuẩn giữa 10 phần: 0,0327, nhỏ nhất trong sáu thuật toán. Việc lấy trung bình phiếu của nhiều cây làm kết quả ổn định hơn giữa các lần chia dữ liệu, và đó mới là lợi ích thấy được rõ nhất của phương pháp kết hợp trên tập dữ liệu này.

AdaBoost với cấu hình mặc định đạt Accuracy = 94,67% và Kappa = 0,9200, thấp nhất trong sáu thuật toán, kèm độ lệch chuẩn 0,0653, lớn nhất trong sáu. Mỗi mô hình thành phần chỉ là một cây một tầng, xét đúng một ngưỡng trên một đặc trưng, nên rất yếu; AdaBoost bù lại bằng cách gộp 50 mô hình như vậy theo trọng số, và kết quả vẫn trên 94%. Bài học quan trọng ở đây là phương pháp kết hợp không tự động cho kết quả tốt hơn thuật toán đơn lẻ: trên Iris, bagging nhích lên một mẫu còn boosting tụt một mẫu so với cây đơn lẻ. Cả hai phương pháp đều phân lớp đúng toàn bộ setosa và chỉ nhầm trong vùng chồng lấn versicolor – virginica, đúng như các thuật toán khác.

### Mở rộng: Stacking

Stacking (Mục 4.2.5) gộp nhiều thuật toán khác loại bằng một bộ phân lớp cấp hai học trên dự đoán của chúng. Người học chạy đoạn mã sau, ghi lại Accuracy và Kappa thu được rồi điền vào bảng tổng hợp của mình để so với sáu thuật toán đã chạy.



In [ ]:
# Can chay khoi ma Chuan bi truoc
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

co_so = [
    ('knn', make_pipeline(StandardScaler(),
                          KNeighborsClassifier(n_neighbors=5))),
    ('nb', GaussianNB()),
    ('cay', DecisionTreeClassifier(random_state=42)),
]
stk = StackingClassifier(estimators=co_so,
                         final_estimator=LogisticRegression(max_iter=1000))
danh_gia('Stacking', stk)


### So sánh hiệu năng các thuật toán

Sau khi chạy xong năm tiểu mục trên, tổng hợp kết quả của cả sáu thuật toán vào một bảng. Mọi dòng trong bảng đều dùng cùng tập dữ liệu Iris và cùng cách đánh giá là kiểm chứng chéo 10 phần, nên so sánh được với nhau.

Bảng 4.13. Kết quả của 6 thuật toán trên Iris (kiểm chứng chéo 10 phần)

### Nhận xét tổng hợp

Trên tập dữ liệu đơn giản như Iris, cả sáu thuật toán đều nằm trong khoảng 94,67% đến 96,00%, nghĩa là toàn bộ khoảng cách giữa thuật toán cao nhất và thấp nhất chỉ là hai mẫu trong 150. Kappa đi cùng chiều với Accuracy, từ 0,9200 đến 0,9400, và không cho kết luận nào khác. Khoảng cách đó nhỏ hơn mức dao động giữa các phần của chính một thuật toán (độ lệch chuẩn từ 0,0327 đến 0,0653), nên không thể khẳng định một thuật toán vượt trội chỉ từ bảng này; muốn kết luận thì phải dùng kiểm định thống kê trên nhiều lần chia dữ liệu, như Chương 6 trình bày.

Ba quan sát có giá trị hơn việc xếp hạng. Thứ nhất, cả sáu thuật toán đều phân lớp đúng gần như toàn bộ setosa và chỉ sai trong vùng chồng lấn giữa versicolor và virginica: giới hạn ở đây thuộc về dữ liệu, không thuộc về thuật toán. Thứ hai, bốn thuật toán cùng đạt Accuracy 95,33% nhưng chỉ cho ba ma trận nhầm lẫn khác nhau: Naive Bayes và cây quyết định sai đúng cùng một kiểu, KNN đổi chiều nhầm lẫn giữa hai lớp chồng lấn, còn mạng nơ-ron mất thêm một mẫu setosa. Vì vậy tiêu chí chọn mô hình phải tính đến loại lỗi nào quan trọng với bài toán, không chỉ tính tổng số mẫu sai. Thứ ba, lựa chọn thuật toán trong thực tế phụ thuộc khả năng diễn giải, tốc độ huấn luyện và độ ổn định chứ không chỉ Accuracy; sự khác biệt về hiệu năng giữa các thuật toán chỉ rõ rệt trên các bài toán phức tạp hơn như ảnh, văn bản hay dữ liệu nhiều chiều.

### Vì sao các con số khác bản chạy trên WEKA

Tệp 03-thuc-hanh-chuong-4.md trong thư mục weka của kho học phần chạy đúng sáu thuật toán này trên WEKA và cho J48 96,00%, NaiveBayes 96,00% và MultilayerPerceptron 97,33%, trong khi bản Python cho cả ba đều 95,33%. Chênh lệch này không phải lỗi của bên nào, mà do bốn nguyên nhân thuộc về cài đặt. Thứ nhất, cùng một cái tên có thể là hai thuật toán khác nhau: J48 là cài đặt của C4.5, còn DecisionTreeClassifier là cài đặt của CART; hai thuật toán khác nhau ở tiêu chí chọn phép chia và ở cách cắt tỉa nên không buộc phải cho cùng một cây. Thứ hai, cùng một thuật toán có thể khác ở cách mô hình hóa thuộc tính số: NaiveBayes của WEKA ước lượng phân phối của thuộc tính số bằng bộ ước lượng riêng và còn có tùy chọn rời rạc hóa thuộc tính số, còn GaussianNB luôn giả định phân phối chuẩn. Thứ ba, mạng nơ-ron ở hai bên khác cả hàm kích hoạt lẫn thuật toán tối ưu: MultilayerPerceptron dùng nơ-ron sigmoid huấn luyện bằng lan truyền ngược có quán tính, còn MLPClassifier dùng ReLU ở tầng ẩn với thuật toán tối ưu Adam. Thứ tư, hai công cụ chia mười phần theo hai cách khác nhau: WEKA xáo trộn dữ liệu theo seed trước khi chia, còn cv=10 của scikit-learn chia phân tầng mà không xáo trộn.

Trên 150 mẫu, mỗi mẫu bị phân lớp sai ứng với 0,67 điểm phần trăm, nên khoảng cách giữa 95,33% và 97,33% chỉ là ba mẫu, nhỏ hơn mức dao động giữa các phần của chính một thuật toán. Chênh lệch cũng không đến từ dữ liệu: tệp iris.arff đi kèm WEKA theo bản UCI còn hai giá trị sai ở mẫu 35 và 38 so với bản load_iris đã sửa theo Fisher, và việc đối chiếu cho thấy hai mẫu này không làm đổi con số nào trong Bảng 4.13. Bài học rút ra là một con số hiệu năng chỉ có nghĩa khi đi kèm cài đặt thuật toán, tham số và cách chia dữ liệu; khi so sánh các thuật toán với nhau, phải so trong cùng một công cụ và cùng một cách đánh giá, đúng như Bảng 4.13 đã làm.

### Những lưu ý khi đối chiếu kết quả

### Lưu ý 1: các con số trong mục này tái lập được

Vì cv=10 chia phân tầng mà không xáo trộn, cách chia mười phần là xác định, không phụ thuộc số ngẫu nhiên. Người học chạy đúng các đoạn mã trên, với scikit-learn 1.8.x, sẽ thu được đúng các con số ở các khối "Đối chiếu kết quả". Cây quyết định, Random Forest, AdaBoost và mạng nơ-ron có thành phần ngẫu nhiên bên trong, nên phải giữ random_state=42 như trong mã.

### Lưu ý 2: khi kết quả không khớp

- Kiểm tra phiên bản scikit-learn; các giá trị mặc định của một số lớp có thể đổi giữa các phiên bản lớn.

- Kiểm tra không thêm shuffle=True hay một KFold tự khai báo vào chỗ cv=10.

- Kiểm tra StandardScaler nằm trong Pipeline, không áp dụng một lần cho toàn bộ dữ liệu: cách làm sau

gây rò rỉ dữ liệu và thường cho con số đẹp hơn thực tế.

- Nhớ rằng lệch một mẫu chỉ là 0,67 điểm phần trăm; chênh lệch ở chữ số thập phân thứ tư thường do làm

tròn, nhưng lệch số mẫu sai thì phải tìm nguyên nhân.

### Gợi ý bài thực hành mở rộng

